# Checkpoint Prediction — Event Extraction

Notebook dùng để **generate prediction trên validation set bằng nhiều LoRA checkpoint** (thường là checkpoint theo epoch), sau đó lưu từng checkpoint thành một file JSONL riêng.

## Các chức năng

1. **Chia checkpoint cho nhiều notebook/account:** chạy tất cả, hoặc chia thành nhóm `A` và `B` theo thứ tự checkpoint.
2. **Resume prediction:** nếu file output đã tồn tại, notebook đọc các `id` đã có và chỉ predict những sample còn thiếu.
3. **Đổi dataset linh hoạt:** khai báo nhiều dataset trong `DATA_CONFIGS`, sau đó chọn bằng `DATA_KEY`.
4. **Bỏ qua checkpoint đã xử lý:** khai báo step trong `PROCESSED_CHECKPOINTS`.

> Mỗi checkpoint tạo một file `val_checkpoint-<step>.jsonl`. Không nên dùng chung `OUTPUT_DIR` cho các lần chạy có dataset khác nhau nếu muốn giữ kết quả riêng; cấu hình bên dưới tự thêm `DATA_KEY` vào thư mục output.

## 1. Cài đặt thư viện

Chạy cell này khi môi trường Kaggle chưa có `bitsandbytes` phù hợp.

In [ ]:
%pip install -U "bitsandbytes>=0.46.1"

## 2. Import thư viện

In [ ]:
import json
import os
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

## 3. Cấu hình

**Những phần thường cần thay đổi được đánh dấu `# CHANGE`.**

- `BASE_MODEL_PATH`: đường dẫn base model trong Kaggle Input.
- `CHECKPOINT_ROOT`: thư mục chứa các checkpoint đã upload.
- `DATA_CONFIGS`: khai báo đường dẫn dataset và file validation.
- `DATA_KEY`: chọn dataset muốn chạy.
- `SPLIT_MODE`: `all`, `A` hoặc `B`.
- `PROCESSED_CHECKPOINTS`: các checkpoint đã predict xong và muốn bỏ qua.
- `MAX_SAMPLES`: giới hạn số sample để chạy thử; đặt `None` để chạy toàn bộ.

In [ ]:
# =========================
# MODEL / CHECKPOINT CONFIG
# =========================

BASE_MODEL_PATH = "/kaggle/input/datasets/md0612/qwen3-4b"  # CHANGE: base model

CHECKPOINT_ROOT = Path(
    "/kaggle/input/models/md0612/qwen3-4b-qlora-vhe-all-checkpoint/pytorch/default/5"
)  # CHANGE: thư mục chứa các checkpoint

# =========================
# DATA CONFIG
# =========================

DATA_CONFIGS = {
    "vhe": {
        "data_path": "/kaggle/input/datasets/md0612/vhe-adapted",  # CHANGE: dataset directory
        "valid_file": "dev.jsonl",  # CHANGE: validation filename
    },
    # Có thể thêm dataset khác, ví dụ:
    # "bkee": {
    #     "data_path": "/kaggle/input/datasets/md0612/bkee-adapted",
    #     "valid_file": "dev.jsonl",
    # },
}

DATA_KEY = "vhe"  # CHANGE: chọn key tương ứng trong DATA_CONFIGS

# =========================
# CHECKPOINT SELECTION
# =========================

SPLIT_MODE = "all"  # CHANGE: "all", "A", hoặc "B"

# Các checkpoint đã predict hoàn tất và muốn bỏ qua.
# Điền số step, không điền chuỗi "checkpoint-...".
PROCESSED_CHECKPOINTS = {
    409,
    1227,
    2045,
    2863,
    3681,
}

# =========================
# OUTPUT / INFERENCE CONFIG
# =========================

OUTPUT_ROOT = Path("/kaggle/working/ee_valid_results")  # CHANGE: thư mục output gốc
MAX_SAMPLES = None  # CHANGE: None = toàn bộ; ví dụ 20 = chạy thử 20 samples

MAX_NEW_TOKENS = 256

## 4. Khởi tạo cấu hình runtime

Cell này kiểm tra `DATA_KEY`, tạo thư mục output riêng cho dataset và cấu hình quantization 4-bit.

In [ ]:
if DATA_KEY not in DATA_CONFIGS:
    raise KeyError(
        f"DATA_KEY={DATA_KEY!r} không tồn tại. "
        f"Các lựa chọn: {list(DATA_CONFIGS)}"
    )

DATA_PATH = Path(DATA_CONFIGS[DATA_KEY]["data_path"])
VALID_SET = DATA_CONFIGS[DATA_KEY]["valid_file"]
VALID_PATH = DATA_PATH / VALID_SET

OUTPUT_DIR = OUTPUT_ROOT / DATA_KEY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not VALID_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy validation file: {VALID_PATH}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Dataset      : {DATA_KEY}")
print(f"Validation   : {VALID_PATH}")
print(f"Output       : {OUTPUT_DIR}")
print(f"Split mode   : {SPLIT_MODE}")

## 5. Load tokenizer, base model và validation data

Base model chỉ load **một lần**. Trong vòng lặp checkpoint, mỗi LoRA adapter được nạp, sử dụng rồi xóa trước khi chuyển sang checkpoint tiếp theo.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)

with VALID_PATH.open("r", encoding="utf-8") as f:
    valid_data = [json.loads(line) for line in f if line.strip()]

if MAX_SAMPLES is not None:
    valid_data = valid_data[:MAX_SAMPLES]

print(f"Number of validation samples: {len(valid_data)}")

## 6. Tìm và chia checkpoint

Quy tắc chia:

- `all`: lấy toàn bộ checkpoint hợp lệ.
- `A`: lấy checkpoint ở vị trí `0, 2, 4, ...`.
- `B`: lấy checkpoint ở vị trí `1, 3, 5, ...`.

Việc chia được thực hiện **sau khi sắp xếp checkpoint theo step tăng dần**. Vì vậy, hai notebook dùng cùng `CHECKPOINT_ROOT`, `PROCESSED_CHECKPOINTS` và chọn `A`/`B` sẽ nhận hai nhóm không trùng nhau.

In [ ]:
if SPLIT_MODE not in {"all", "A", "B"}:
    raise ValueError("SPLIT_MODE chỉ nhận 'all', 'A' hoặc 'B'.")

if not CHECKPOINT_ROOT.is_dir():
    raise FileNotFoundError(f"Không tìm thấy CHECKPOINT_ROOT: {CHECKPOINT_ROOT}")

all_checkpoints = sorted(
    [
        p for p in CHECKPOINT_ROOT.iterdir()
        if p.is_dir()
        and p.name.startswith("checkpoint-")
        and p.name.removeprefix("checkpoint-").isdigit()
        and int(p.name.removeprefix("checkpoint-")) not in PROCESSED_CHECKPOINTS
        and (p / "adapter_config.json").is_file()
        and (p / "adapter_model.safetensors").is_file()
    ],
    key=lambda p: int(p.name.removeprefix("checkpoint-")),
)

if SPLIT_MODE == "A":
    checkpoints = all_checkpoints[::2]
elif SPLIT_MODE == "B":
    checkpoints = all_checkpoints[1::2]
else:
    checkpoints = all_checkpoints

if not checkpoints:
    raise FileNotFoundError(
        f"Không có checkpoint phù hợp trong {CHECKPOINT_ROOT}. "
        "Kiểm tra đường dẫn, PROCESSED_CHECKPOINTS và SPLIT_MODE."
    )

print(f"Found {len(all_checkpoints)} eligible checkpoints before split.")
print(f"Selected {len(checkpoints)} checkpoints for mode {SPLIT_MODE}:")
for checkpoint in checkpoints:
    print(" -", checkpoint.name)

## 7. Hàm generate

Chỉ decode phần token được sinh thêm, không decode lại prompt đầu vào.

In [ ]:
def generate_prediction(sentence: str, model, tokenizer) -> str:
    inputs = tokenizer(
        sentence,
        add_special_tokens=True,
        return_tensors="pt",
    )

    # Đưa input lên cùng device với embedding layer.
    input_device = model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][prompt_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

## 8. Đọc các sample đã hoàn thành

Notebook dùng `id` trong JSONL làm khóa resume. Nếu notebook bị ngắt giữa chừng, lần chạy sau sẽ bỏ qua các `id` đã ghi và tiếp tục phần còn thiếu.

> Điều kiện: `id` phải duy nhất trong validation set và ổn định giữa các lần chạy.

In [ ]:
def load_completed_predictions(output_path: Path) -> set:
    completed_ids = set()

    if not output_path.exists():
        return completed_ids

    with output_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                # Bỏ qua dòng JSON lỗi (ví dụ notebook bị ngắt lúc đang ghi).
                continue

            if "id" in record:
                completed_ids.add(record["id"])

    return completed_ids

## 9. Predict một checkpoint và tự resume

- Mỗi checkpoint có một file output riêng.
- Mở file ở chế độ append (`"a"`), không xóa kết quả cũ.
- Sau mỗi sample, ghi và `flush()` để giảm nguy cơ mất kết quả khi session bị ngắt.
- Nếu tất cả sample đã có `id` trong file, checkpoint đó được bỏ qua.

In [ ]:
def evaluate_checkpoint(
    checkpoint: Path,
    valid_data: list,
    model,
    tokenizer,
    output_dir: Path,
) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"val_{checkpoint.name}.jsonl"

    completed_ids = load_completed_predictions(output_path)
    valid_ids = {sample.get("id", "") for sample in valid_data}
    total_completed = len(valid_ids & completed_ids)
    total_samples = len(valid_data)

    print(f"\\n===== {checkpoint.name} =====")
    print(f"Completed: {total_completed}/{total_samples}")

    if total_completed == total_samples:
        print(f"Already complete: {output_path}")
        return output_path

    model.eval()

    with output_path.open("a", encoding="utf-8") as out:
        for sample in valid_data:
            sample_id = sample.get("id", "")

            if sample_id in completed_ids:
                continue

            sentence = sample.get("text", "")
            result = generate_prediction(sentence, model, tokenizer)

            record = {
                "id": sample_id,
                "result": result,
            }

            out.write(json.dumps(record, ensure_ascii=False) + "\\n")
            out.flush()
            completed_ids.add(sample_id)

            done = len(valid_ids & completed_ids)
            if done % 10 == 0 or done == total_samples:
                print(
                    f"\\rProgress: {done}/{total_samples}",
                    end="",
                    flush=True,
                )

    print(f"\\nSaved: {output_path}")
    return output_path

## 10. Chạy lần lượt tất cả checkpoint

Lưu ý quan trọng: mỗi vòng lặp đều nạp adapter với tên `current`, chạy inference, sau đó xóa adapter đó. Cách này tránh lỗi trùng tên adapter khi chuyển checkpoint.

Nếu một checkpoint đã có file JSONL đầy đủ, hàm `evaluate_checkpoint` tự bỏ qua phần generate.

In [ ]:
def run_all_checkpoints(
    checkpoints: list,
    valid_data: list,
    model,
    tokenizer,
    output_dir: Path,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    for index, checkpoint in enumerate(checkpoints, start=1):
        print(
            f"\\n===== {checkpoint.name} "
            f"({index}/{len(checkpoints)}) ====="
        )

        # Nạp adapter của checkpoint hiện tại.
        model.load_adapter(
            str(checkpoint),
            adapter_name="current",
        )
        model.set_adapter("current")

        try:
            evaluate_checkpoint(
                checkpoint=checkpoint,
                valid_data=valid_data,
                model=model,
                tokenizer=tokenizer,
                output_dir=output_dir,
            )
        finally:
            # Luôn xóa adapter, kể cả khi inference gặp lỗi.
            model.delete_adapter("current")

    print("\\nAll selected checkpoint predictions completed.")
    print(f"Results directory: {output_dir}")

## 11. Chạy inference

Chạy cell này sau khi các cell cấu hình, load model/data và định nghĩa hàm đã chạy thành công.

In [ ]:
run_all_checkpoints(
    checkpoints=checkpoints,
    valid_data=valid_data,
    model=model,
    tokenizer=tokenizer,
    output_dir=OUTPUT_DIR,
)

## 12. Kiểm tra kết quả

Mỗi file tương ứng với một checkpoint. Có thể kiểm tra nhanh số dòng và số `id` duy nhất.

In [ ]:
for checkpoint in checkpoints:
    output_path = OUTPUT_DIR / f"val_{checkpoint.name}.jsonl"

    if not output_path.exists():
        print(f"{checkpoint.name}: output file not found")
        continue

    records = []
    with output_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

    unique_ids = {record.get("id") for record in records}
    print(
        f"{checkpoint.name}: "
        f"{len(records)} records, {len(unique_ids)} unique IDs"
    )

---

## Hướng dẫn các tình huống sử dụng

### Chạy toàn bộ checkpoint
```python
SPLIT_MODE = "all"
```

### Chia cho hai notebook
Notebook thứ nhất:
```python
SPLIT_MODE = "A"
```
Notebook thứ hai:
```python
SPLIT_MODE = "B"
```
Hai notebook cần dùng cùng danh sách checkpoint và cùng `PROCESSED_CHECKPOINTS`.

### Chạy dataset khác
Thêm cấu hình vào `DATA_CONFIGS`, sau đó đổi `DATA_KEY`. Ví dụ:
```python
DATA_CONFIGS["geneva"] = {
    "data_path": "/kaggle/input/datasets/md0612/geneva-adapted",
    "valid_file": "dev.jsonl",
}
DATA_KEY = "geneva"
```

### Chạy thử một phần validation set
```python
MAX_SAMPLES = 20
```
Để chạy toàn bộ:
```python
MAX_SAMPLES = None
```

### Đánh dấu checkpoint đã xử lý
Thêm step vào `PROCESSED_CHECKPOINTS`, ví dụ:
```python
PROCESSED_CHECKPOINTS = {409, 1227, 2045}
```

### Resume sau khi Kaggle session bị ngắt
Giữ nguyên `OUTPUT_DIR` và chạy lại notebook. Các `id` đã có trong file JSONL sẽ được bỏ qua.

**Lưu ý:** Nếu đổi dataset, nên dùng `DATA_KEY` khác để output nằm ở thư mục riêng. Nếu đổi nội dung validation set nhưng giữ nguyên `id`, hãy xóa output cũ hoặc đổi `OUTPUT_ROOT` để tránh resume nhầm dữ liệu.